# Themis Judge Model — SFT Training
Fine-tuning Llama 3.2-3B on moot court judge data using Unsloth + LoRA

## Step 1: Install Dependencies

In [ ]:
!pip install unsloth
!pip install datasets trl transformers accelerate
!pip install huggingface_hub

## Step 2: HuggingFace Login

In [ ]:
from huggingface_hub import login

# Add your HF token as a Kaggle secret named HF_TOKEN
# Kaggle → Add-ons → Secrets → Add Secret
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)

## Step 3: Load Dataset from HuggingFace

In [ ]:
from datasets import load_dataset

dataset = load_dataset('YOUR_HF_USERNAME/themis-judge-sft', split='train')
print(f'Total examples: {len(dataset)}')
print(dataset[0])

## Step 4: Format Dataset into Prompt Strings

In [ ]:
import json

prompt_template = """### Case Summary:
{case_summary}

### Legal Issue:
{legal_issue}

### Relevant Laws:
{relevant_laws}

### Side:
{side}

### Opposing Counsel Argument:
{opposing_last_argument}

### Previous Judge Response:
{judge_last_response}

### CurrentArgument:
{current_argument}

### Judge Response:
{output}{eos_token}"""


def format_sample(sample, eos_token):
    inp = sample['input']
    prompt = prompt_template.format(
        case_summary=inp['case_summary'],
        legal_issue=inp['legal_issue'],
        relevant_laws=inp['relevant_laws'] if isinstance(inp['relevant_laws'], str) else ', '.join(inp['relevant_laws']),    # if in list format, convert into string
        side=inp['side'].upper(),
        opposing_last_argument=inp.get('opposing_last_argument', 'None'),
        judge_last_response=inp.get('judge_last_response', 'None'),
        current_argument=inp['current_argument'],
        output=json.dumps(sample['output'], ensure_ascii=False),    # JSON string output
        eos_token=eos_token
    ) 
    return {'text': prompt}


print('Formatted Sample\n')
sample = format_sample(dataset[0], eos_token='/s')
print(sample['text'])

## Step 5: Load Llama 3.2-3B with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # auto-detect
    load_in_4bit=True,    # 4-bit quantization for memory efficiency
)
``
EOS_TOKEN = tokenizer.eos_token
print(f'EOS token: {EOS_TOKEN}')
print(f'Model loaded. Vocab size: {tokenizer.vocab_size}')

## Step 6: Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print('LoRA adapters added.')
model.print_trainable_parameters()

## Step 7: Format and Split Dataset

In [ ]:
# Now apply formatting with real EOS token
formatted = dataset.map(
    lambda ex: format_sample(ex, EOS_TOKEN),
    remove_columns=dataset.column_names  # keep only 'text'
)

# Train/eval split — 90/10
split = formatted.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f'Train: {len(train_dataset)} examples')
print(f'Eval: {len(eval_dataset)} examples')
print(f'\nSample prompt length: {len(train_dataset[0]["text"])} chars')

## Step 8: Enable Training Mode

In [ ]:
FastLanguageModel.for_training(model)
print('Model set to training mode.')

## Step 9: Configure and Run Trainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_steps=10,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        output_dir='outputs',
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        report_to='none',
    )
)

print('Trainer configured. Starting training...')
trainer_stats = trainer.train()
print(f'Training complete. Runtime: {trainer_stats.metrics["train_runtime"]:.1f}s')

## Step 10: Check Loss Curve

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps = [x['step'] for x in log_history if 'loss' in x]
train_loss = [x['loss'] for x in log_history if 'loss' in x]
eval_loss = [x['eval_loss'] for x in log_history if 'eval_loss' in x]
eval_steps = [x['step'] for x in log_history if 'eval_loss' in x]

plt.figure(figsize=(10, 4))
plt.plot(train_steps, train_loss, label='Train Loss')
plt.plot(eval_steps, eval_loss, label='Eval Loss', marker='o')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.legend()
plt.grid(True)
plt.show()

print(f'Final train loss: {train_loss[-1]:.4f}')
print(f'Final eval loss: {eval_loss[-1]:.4f}')
print('\nInterpretation:')
print('- Both going down: good')
print('- Train down, eval up: overfitting')
print('- Both flat: learning rate too low or data issue')

## Step 11: Quick Inference Test

In [ ]:
FastLanguageModel.for_inference(model)

# Use a sample from eval set
test_example = eval_dataset[0]['text']
# Strip everything after '### Judge Response:' — that's what model should generate
prompt_only = test_example.split('### Judge Response:')[0] + '### Judge Response:\n'

inputs = tokenizer(prompt_only, return_tensors='pt').to('cuda')

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Model output:')
print(generated)
print('\nExpected output:')
print(test_example.split('### Judge Response:')[1])

## Step 12: Save LoRA Adapter to HuggingFace

In [ ]:
REPO_NAME = 'YOUR_HF_USERNAME/themis-judge-lora'

model.push_to_hub(REPO_NAME, token=hf_token)
tokenizer.push_to_hub(REPO_NAME, token=hf_token)

print(f'LoRA adapter saved to: https://huggingface.co/{REPO_NAME}')

## Step 13: Save Merged Model (Optional — for GGUF later)

In [ ]:
# Merge LoRA into base model and save
# Only run this if you have enough disk space (~6GB)
# Skip for now if just testing — LoRA adapter is enough

MERGED_REPO = 'YOUR_HF_USERNAME/themis-judge-merged'

model.push_to_hub_merged(
    MERGED_REPO,
    tokenizer,
    save_method='merged_16bit',
    token=hf_token
)

print(f'Merged model saved to: https://huggingface.co/{MERGED_REPO}')

## Step 14: Save GGUF (for local testing or deployment)

In [ ]:
# Quantize and save as GGUF — Q4_K_M is best quality/size tradeoff
GGUF_REPO = 'YOUR_HF_USERNAME/themis-judge-gguf'

model.push_to_hub_gguf(
    GGUF_REPO,
    tokenizer,
    quantization_method='q4_k_m',
    token=hf_token
)

print(f'GGUF model saved to: https://huggingface.co/{GGUF_REPO}')